# **1. Perkenalan Dataset**

**Nama Siswa**: Bryan Dewa Wicaksana  
**Proyek**: Membangun Sistem Machine Learning (MSML) - Submission  
**Dataset**: Heart Disease Prediction Dataset (UCI Machine Learning Repository)  

### **Deskripsi & Latar Belakang**
Penyakit jantung (Heart Disease) merupakan salah satu penyebab kematian tertinggi di dunia. Eksperimen ini bertujuan untuk mengolah data medis klinis pasien guna membangun modelMachine Learning yang mampu mengklasifikasikan apakah seorang pasien berisiko mengidap penyakit jantung (`target = 1`) atau tidak (`target = 0`).

### **Keterangan Fitur Dataset:**
1. `age`: Usia pasien (tahun)
2. `sex`: Jenis kelamin (1 = Laki-laki, 0 = Perempuan)
3. `cp`: Tipe nyeri dada / Chest Pain (0: Typical Angina, 1: Atypical Angina, 2: Non-anginal Pain, 3: Asymptomatic)
4. `trestbps`: Tekanan darah istirahat (mm Hg saat masuk rumah sakit)
5. `chol`: Kolesterol serum (mg/dl)
6. `fbs`: Gula darah puasa > 120 mg/dl (1 = true; 0 = false)
7. `restecg`: Hasil elektrokardiografi istirahat (0, 1, 2)
8. `thalach`: Detak jantung maksimum yang dicapai
9. `exang`: Angina yang diinduksi olahraga (1 = ya; 0 = tidak)
10. `oldpeak`: Depresi ST yang diinduksi oleh olahraga relatif terhadap istirahat
11. `slope`: Kemiringan segmen ST olahraga puncak
12. `ca`: Jumlah pembuluh darah utama (0-3) yang diwarnai dengan fluorosopi
13. `thal`: Hasil thalassemia (1 = normal; 2 = fixed defect; 3 = reversable defect)
14. `target`: Diagnosis penyakit jantung (0 = Sehat, 1 = Memiliki Penyakit Jantung)

# **2. Import Library**

Pada tahap ini, kita mengimpor seluruh pustaka Python yang dibutuhkan untuk analisis data, visualisasi, preprocessing, dan pemodelan machine learning.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Konfigurasi grafik
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
print('[INFO] Seluruh pustaka berhasil diimpor!')

# **3. Memuat Dataset**

Memuat data mentah (`heart_raw.csv`) ke dalam DataFrame Pandas dan melakukan verifikasi struktur data awal.

In [ ]:
# Memuat data mentah
df_raw = pd.read_csv('heart_raw.csv')
print('=== 5 Baris Pertama Dataset ===')
display(df_raw.head())

print('\n=== Informasi Struktur Dataset ===')
df_raw.info()

print('\n=== Statistik Deskriptif Fitur ===')
display(df_raw.describe())

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita menganalisis karakteristik dataset, distribusi variabel target, hubungan antar fitur, serta pendeteksian pencilan (outliers).

In [ ]:
# 1. Visualisasi Distribusi Target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='target', data=df_raw, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribusi Diagnosis Penyakit Jantung')
axes[0].set_xticklabels(['Sehat (0)', 'Sakit (1)'])
axes[0].set_ylabel('Jumlah Pasien')

df_raw['target'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1], colors=['#2ecc71', '#e74c3c'], labels=['Sehat (0)', 'Sakit (1)'], explode=[0, 0.05])
axes[1].set_title('Persentase Proporsi Target')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# 2. Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df_raw.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriks Korelasi Antar Fitur')
plt.show()

# 3. Boxplot Deteksi Outlier Fitur Numerik
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_raw[num_cols], palette='Set2')
plt.title('Distribusi dan Outlier pada Fitur Kontinu')
plt.show()

# **5. Data Preprocessing**

Tahap preprocessing meliputi:
1. **Pembersihan Data**: Penanganan nilai hilang (missing values) dan duplikat.
2. **Feature Scaling**: Standarisasi fitur kontinu (`StandardScaler`).
3. **Pemisahan Data**: Pembagian data latih (80%) dan data uji (20%).
4. **Penyimpanan Processed Dataset**: Menyimpan hasil preprocessing ke `heart_processed.csv`.

In [ ]:
# 1. Cek Missing Values & Duplikat
print('Jumlah missing value per kolom:')
print(df_raw.isnull().sum())
print('Jumlah duplikat:', df_raw.duplicated().sum())

# Imputasi missing values jika ada
df_clean = df_raw.copy()
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 2. Scaling Fitur Kontinu
X = df_clean.drop(columns=['target'])
y = df_clean['target']

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Gabungkan kembali
df_processed = pd.concat([X, y.reset_index(drop=True)], axis=1)
df_processed.to_csv('heart_processed.csv', index=False)
print('\n[SUCCESS] Preprocessing selesai. Processed dataset disimpan ke heart_processed.csv')
display(df_processed.head())

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Bentuk X_train: {X_train.shape}, X_test: {X_test.shape}')

# **6. Eksperimen Pelatihan Model Baseline**

Melatih model awal (Random Forest Classifier) pada dataset terproses untuk mengukur performa baseline.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Akurasi Baseline Random Forest: {acc:.4f} ({acc*100:.2f}%)\n')
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Sehat', 'Sakit']))